# 分類器


In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import mne
from scipy.ndimage import uniform_filter1d
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
mne.set_log_level('WARNING')


In [2]:
PATH_HC = 'C:/Users/kitt9/OneDrive - National ChengChi University/eeg-rag-llm-diag/origin_data_set/HC/'
PATH_MCI = 'C:/Users/kitt9/OneDrive - National ChengChi University/eeg-rag-llm-diag/origin_data_set/MCI/'
PATH_AD = 'C:/Users/kitt9/OneDrive - National ChengChi University/eeg-rag-llm-diag/origin_data_set/AD/'

# 特徵提取

# Alpha 頻帶

In [3]:
def calculate_paf(psd_values, freqs, alpha_range=(4, 12)):
    """計算 Alpha 頻帶峰值頻率 (PAF)"""
    psd_values = uniform_filter1d(psd_values, size=5)
    alpha_idx = np.where((freqs >= alpha_range[0]) & (freqs <= alpha_range[1]))[0]
    alpha_psd = psd_values[alpha_idx]
    peak_idx = np.argmax(alpha_psd)
    return freqs[alpha_idx][peak_idx]

def calculate_cgf(psd_values, freqs, alpha_range=(4, 12)):
    """計算 Alpha 頻帶重心頻率 (CGF)"""
    psd_values = uniform_filter1d(psd_values, size=5)
    alpha_idx = np.where((freqs >= alpha_range[0]) & (freqs <= alpha_range[1]))[0]
    alpha_psd = psd_values[alpha_idx]
    if np.sum(alpha_psd) == 0:
        return np.nan
    return np.sum(freqs[alpha_idx] * alpha_psd) / np.sum(alpha_psd)

def calculate_af_amplitude(psd_values, freqs, af):
    """計算指定頻率點的振幅值"""
    return np.interp(af, freqs, psd_values)


# HFD

In [4]:
def calculate_hfd(data, k_max=10):
    """計算單一通道時域訊號的 Higuchi Fractal Dimension (HFD)"""
    N = len(data)
    L_k = np.zeros(k_max)
    for k in range(1, k_max + 1):
        Lk_m = []
        for m in range(k):
            indices = np.arange(m, N, k)
            n_max = len(indices)
            if n_max > 1:
                diffs = np.abs(np.diff(data[indices]))
                l_m_k = (np.sum(diffs) * (N - 1) / ((n_max - 1) * k)) / k
                Lk_m.append(l_m_k)
        L_k[k-1] = np.mean(Lk_m) if Lk_m else 0
    valid = L_k > 0
    if not np.any(valid): 
        return np.nan
    x = np.log(1.0 / np.arange(1, k_max + 1)[valid])
    y = np.log(L_k[valid])
    hfd, _ = np.polyfit(x, y, 1)
    return hfd

# 讀取

In [5]:
def process_eeg_directory(path, group_name, event_name='Eyes Closed', delay_time=0, cut_time=8, k_max=10):
    """
    讀取指定資料夾內的所有 EDF 檔案，同時計算 HFD 與 PSD 特徵並返回 DataFrame
    """
    if not os.path.exists(path):
        print(f"錯誤：找不到路徑 {path}")
        return pd.DataFrame()
        
    filesname = os.listdir(path)
    edf_files = [f for f in filesname if f.lower().endswith('.edf')]
    
    bands = {
        'Delta': (0.5, 4),
        'Theta': (4, 8),
        'Alpha': (8, 12)
    }
    
    group_features = []
    print(f"\n[開始處理] 組別: {group_name}, 預計處理檔案數: {len(edf_files)}")
    
    for filename in edf_files:
        file_path = os.path.join(path, filename)
        try:
            raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
        except Exception as e:
            print(f"  ❌ 檔案 {filename} 讀取失敗: {e}")
            continue

        # 提取標註事件
        try:
            events, event_id = mne.events_from_annotations(raw, verbose=False)
            s_events = events[events[:, 2] == event_id[event_name]]
        except Exception as e:
            print(f"  ⚠️ 檔案 {filename} 未包含事件 '{event_name}': {e}")
            continue

        if 'Photic' in raw.info['ch_names']:
            raw.drop_channels(['Photic'])

        fs = int(raw.info['sfreq'])
        filter_low, filter_high = 0.5, 50
        ch_names = raw.info['ch_names']
        ch_types = ['eeg'] * len(ch_names)

        # 暫存當前受試者所有 epoch 的數據
        subject_psd_results = {ch: [] for ch in ch_names}
        subject_hfd_results = {ch: [] for ch in ch_names}
        freqs = None

        # 遍歷每個事件區段並提取時頻訊號
        for s_event in s_events:
            event_start = s_event[0]
            start_sample = event_start + int(fs * delay_time)
            end_sample = start_sample + int(fs * cut_time)

            if end_sample > len(raw): 
                continue

            event_data, _ = raw[:, start_sample:end_sample]
            event_data_filtered = mne.filter.filter_data(event_data, sfreq=fs, l_freq=filter_low, h_freq=filter_high, verbose=False)
            
            info = mne.create_info(ch_names=ch_names, sfreq=fs, ch_types=ch_types)
            epoch_raw = mne.io.RawArray(event_data_filtered, info, verbose=False)
            epoch_data_clean = epoch_raw.get_data()

            # 計算當前 epoch 各通道的特徵
            for ch_idx, ch_name in enumerate(ch_names):
                # 1. 頻域 PSD 特徵計算
                psd, freqs = mne.time_frequency.psd_array_welch(epoch_data_clean[ch_idx], 
                                                                sfreq=fs, fmin=filter_low, fmax=filter_high, 
                                                                n_fft=int(fs*2), verbose=False)
                psd = psd * (1e6 ** 2)  # 單位轉換
                subject_psd_results[ch_name].append(psd)
                
                # 2. 時域 HFD 複雜度計算
                hfd_val = calculate_hfd(epoch_data_clean[ch_idx], k_max=k_max)
                subject_hfd_results[ch_name].append(hfd_val)

        # 確保有成功提取到 epoch 數據
        if len(subject_psd_results[ch_names[0]]) == 0:
            print(f"  ⚠️ 檔案 {filename} 無有效 Epoch 區段")
            continue

        # 對所有 Epoch 取平均值作為該受試者的特徵
        feature_dict = {'Filename': filename, 'Group': group_name}
        
        for ch_name in ch_names:
            # 整合 HFD 平均值
            feature_dict[f'{ch_name}'] = np.mean(subject_hfd_results[ch_name])
            
            # 整合 PSD 相關特徵
            mean_psd_linear = np.mean(subject_psd_results[ch_name], axis=0)
            mean_psd_db = 10 * np.log10(mean_psd_linear)
            
            # 各主要頻帶功率 (dB)
            for band_name, band_range in bands.items():
                band_idx = [i for i, f in enumerate(freqs) if band_range[0] <= f < band_range[1]]
                feature_dict[f'{ch_name}_{band_name}_Power'] = np.mean(mean_psd_db[band_idx])
            
            # Alpha 核心指標：PAF, CGF, Amplitude
            paf = calculate_paf(mean_psd_linear, freqs)
            cgf = calculate_cgf(mean_psd_linear, freqs)
            paf_amp = 10 * np.log10(calculate_af_amplitude(mean_psd_linear, freqs, paf))
            
            feature_dict[f'{ch_name}_PAF'] = paf
            feature_dict[f'{ch_name}_CGF'] = cgf
            feature_dict[f'{ch_name}_Alpha_Amplitude'] = paf_amp

        group_features.append(feature_dict)
        
    return pd.DataFrame(group_features)

# Feature Integration

In [6]:
if __name__ == "__main__":
    # 確保所有 Scikit-Learn 工具與系統套件完整宣告
    from sklearn.preprocessing import StandardScaler, LabelEncoder
    from sklearn.model_selection import StratifiedKFold
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
    from sklearn.metrics.pairwise import cosine_similarity
    import shutil
    import os
    
    # 1. 批量讀取與提取特徵
    df_hc = process_eeg_directory(PATH_HC, 'HC')
    df_mci = process_eeg_directory(PATH_MCI, 'MCI')
    df_ad = process_eeg_directory(PATH_AD, 'AD')
    
    # 檢查是否有任何資料被讀取
    if df_hc.empty and df_mci.empty and df_ad.empty:
        print("❌ 錯誤：未讀取到任何有效的 EEG 生理特徵。請檢查路徑設定與檔案內容。")
        exit()
        
    # 合併為大主表 (Master DataFrame)
    master_df = pd.concat([df_hc, df_mci, df_ad], ignore_index=True)
    print(f"\n[特徵整合完畢] 總樣本數: {len(master_df)} 筆, 總特徵維度: {master_df.shape[1] - 2} 維")
    
    # 2. 特徵欄位定義 (排除 Filename 與 Group)
    feature_columns = [col for col in master_df.columns if col not in ['Filename', 'Group']]
    
    # 3. 依據研究計畫書規範：計算餘弦相似度，並切分前 80% (Sample) 與後 20% (Test)
    typical_group_dfs = []  # 存放前 80% 典型樣本
    remaining_group_dfs = []  # 存放剩下 20% 測試樣本
    
    print("\n" + "="*60)
    print("  各組樣本切分結果 (前 80% 典型隨機 / 後 20% 邊緣測試)")
    print("="*60)
    
    for group, group_df in master_df.groupby('Group'):
        X_group = group_df[feature_columns].values
        
        # 使用 Z-Score 將特徵縮放至同一尺度
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_group)
        
        # 計算該診斷組別在多維特徵空間中的平均特徵向量中心點 (Centroid)
        centroid = np.mean(X_scaled, axis=0).reshape(1, -1)
        
        # 計算組內每個受試者與組中心的餘弦相似度 (Cosine Similarity)
        similarities = cosine_similarity(X_scaled, centroid).flatten()
        
        # 將結果存回暫存表
        temp_df = group_df.copy()
        temp_df['_cosine_similarity'] = similarities
        
        # 找出第 20 百分位數作為切點 (相似度由小到大排序)
        cutoff_value = np.percentile(similarities, 20)
        
        # 篩選分流：大於等於切點為前 80% 典型；小於切點為後 20%
        typical_samples = temp_df[temp_df['_cosine_similarity'] >= cutoff_value].copy()
        remaining_samples = temp_df[temp_df['_cosine_similarity'] < cutoff_value].copy()
        
        # 移除輔助欄位
        typical_samples = typical_samples.drop(columns=['_cosine_similarity'])
        remaining_samples = remaining_samples.drop(columns=['_cosine_similarity'])
        
        typical_group_dfs.append(typical_samples)
        remaining_group_dfs.append(remaining_samples)
        
        print(f"組別 {group:7s}: 原始總數 = {len(group_df):3d} 筆 -> 典型樣本(sample) = {len(typical_samples):3d} 筆 | 剩餘測試(test) = {len(remaining_samples):3d} 筆 (臨界相似度: {cutoff_value:.4f})")
        
    # 整合高質量典型樣本集（後續分類模型仍以此高質量集合進行 5-Fold 驗證）
    high_quality_df = pd.concat(typical_group_dfs, ignore_index=True)
    test_set_df = pd.concat(remaining_group_dfs, ignore_index=True)
    
    # 保存高質量特徵表
    high_quality_df.to_csv("eeg_high_quality_features_cosine.csv", index=False)
    print(f"\n💾 已將篩選後的前 80% 高質量樣本特徵表保存至：eeg_high_quality_features_cosine.csv")
    
    # ==============================================================================
    # 5. 自動創建 sample_test 階層資料夾，並分配實體檔案 (前 80% -> sample, 剩餘 -> test)
    # ==============================================================================
    print("\n" + "="*60)
    print("  自動建立 sample_test 階層資料夾與實體檔案搬移分配")
    print("="*60)
    
    TARGET_ROOT = './sample_test'
    source_paths = {
        'HC': PATH_HC,
        'MCI': PATH_MCI,
        'AD': PATH_AD
    }
    
    # 定義一個輔助函式來處理檔案複製與大小寫相容
    def copy_edf_file(src_dir, filename, target_dir):
        src_file = os.path.join(src_dir, filename)
        dest_file = os.path.join(target_dir, filename)
        
        if os.path.exists(src_file):
            shutil.copy2(src_file, dest_file)
            return True
        else:
            # 應對 Windows/Linux 系統下常見的副檔名大小寫不一致問題 (.edf vs .EDF)
            base_name, ext = os.path.splitext(filename)
            alt_filename = base_name + ext.upper() if ext.islower() else base_name + ext.lower()
            alt_src_file = os.path.join(src_dir, alt_filename)
            if os.path.exists(alt_src_file):
                shutil.copy2(alt_src_file, dest_file)
                return True
        return False

    # A. 處理前 80% 的典型資料夾分流 (放進 sample/)
    sample_copied = 0
    for idx, row in high_quality_df.iterrows():
        group = row['Group']
        filename = row['Filename']
        target_dir = os.path.join(TARGET_ROOT, 'sample', group)
        os.makedirs(target_dir, exist_ok=True)
        if copy_edf_file(source_paths[group], filename, target_dir):
            sample_copied += 1
            
    # B. 處理剩下 20% 的測試資料夾分流 (放進 test/)
    test_copied = 0
    for idx, row in test_set_df.iterrows():
        group = row['Group']
        filename = row['Filename']
        target_dir = os.path.join(TARGET_ROOT, 'test', group)
        os.makedirs(target_dir, exist_ok=True)
        if copy_edf_file(source_paths[group], filename, target_dir):
            test_copied += 1
            
    print(f"🎉 檔案分配架構配置完成！")
    print(f"   📂 [典型黃金庫] -> '{TARGET_ROOT}/sample/' 共成功轉存 {sample_copied} 筆原始腦波檔")
    print(f"   📂 [邊緣測試集] -> '{TARGET_ROOT}/test/'   共成功轉存 {test_copied} 筆原始腦波檔")
    
    # 4. 建立機器學習分類器並進行 5-Fold 分層交叉驗證（基於 sample 的高質量特徵）
    print("\n" + "="*60)
    print("  模型機器學習分類器評估中...")
    print("="*60)
    X = high_quality_df[feature_columns].values
    y = high_quality_df['Group'].values
    
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    all_y_true = []
    all_y_pred = []
    cv_accuracies = []
    
    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y_encoded)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]
        
        # 獨立標準化，防範資料洩漏
        fold_scaler = StandardScaler()
        X_train_scaled = fold_scaler.fit_transform(X_train)
        X_test_scaled = fold_scaler.transform(X_test)
        
        # 建立具備類別權重平衡的隨機森林分類器
        clf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
        clf.fit(X_train_scaled, y_train)
        
        y_pred = clf.predict(X_test_scaled)
        
        cv_accuracies.append(accuracy_score(y_test, y_pred))
        all_y_true.extend(y_test)
        all_y_pred.extend(y_pred)
        
    # 5. 輸出最終分類表現報告
    print("\n" + "="*60)
    print(f"  篩選後高質量樣本(Sample)分類模型評估報告 (5-Fold Cross Validation)")
    print("="*60)
    print(f"交叉驗證平均準確率 (Mean CV Accuracy): {np.mean(cv_accuracies)*100:.2f}%")
    print("-"*60)
    print("精確度與召回率分析 (Classification Report):")
    unique_labels = np.unique(all_y_true)
    print(classification_report(all_y_true, all_y_pred, target_names=le.classes_[unique_labels]))
    print("-"*60)
    print("混淆矩陣 (Confusion Matrix):")
    print(confusion_matrix(all_y_true, all_y_pred))


[開始處理] 組別: HC, 預計處理檔案數: 72
  ⚠️ 檔案 R7729926.edf 未包含事件 'Eyes Closed': 'Eyes Closed'

[開始處理] 組別: MCI, 預計處理檔案數: 88

[開始處理] 組別: AD, 預計處理檔案數: 54
  ⚠️ 檔案 C6444917.edf 未包含事件 'Eyes Closed': 'Eyes Closed'
  ⚠️ 檔案 R3459841.edf 未包含事件 'Eyes Closed': 'Eyes Closed'
  ⚠️ 檔案 R6431046.edf 未包含事件 'Eyes Closed': 'Eyes Closed'

[特徵整合完畢] 總樣本數: 210 筆, 總特徵維度: 126 維

  各組樣本切分結果 (前 80% 典型隨機 / 後 20% 邊緣測試)
組別 AD     : 原始總數 =  51 筆 -> 典型樣本(sample) =  41 筆 | 剩餘測試(test) =  10 筆 (臨界相似度: -0.0369)
組別 HC     : 原始總數 =  71 筆 -> 典型樣本(sample) =  57 筆 | 剩餘測試(test) =  14 筆 (臨界相似度: -0.0637)
組別 MCI    : 原始總數 =  88 筆 -> 典型樣本(sample) =  70 筆 | 剩餘測試(test) =  18 筆 (臨界相似度: -0.1062)

💾 已將篩選後的前 80% 高質量樣本特徵表保存至：eeg_high_quality_features_cosine.csv

  自動建立 sample_test 階層資料夾與實體檔案搬移分配
🎉 檔案分配架構配置完成！
   📂 [典型黃金庫] -> './sample_test/sample/' 共成功轉存 168 筆原始腦波檔
   📂 [邊緣測試集] -> './sample_test/test/'   共成功轉存 42 筆原始腦波檔

  模型機器學習分類器評估中...

  篩選後高質量樣本(Sample)分類模型評估報告 (5-Fold Cross Validation)
交叉驗證平均準確率 (Mean CV Accuracy): 54.80%
--------------------

# 